# Predict — HARNN Word2Vec + Global

Dự đoán nhãn L1 / L2 / L3 cho bài viết mới.

```
Cell 1 → Load model + vocab + label_map
Cell 2 → Hàm predict
Cell 3 → Demo dự đoán
Cell 4 → Batch predict từ file
```


## Cell 1 — Load model & artifacts


In [10]:
# ============================================================
# CELL 1 — Load toàn bộ artifacts cần thiết để predict
# ============================================================
import json, re, torch
import torch.nn as nn
import numpy as np

# ── Đường dẫn — chỉnh lại nếu cần ──────────────────────────────────────
from pathlib import Path
PROJECT_DIR = Path.cwd().parent
CHECKPOINT  = str(PROJECT_DIR / 'output' / 'models' / 'checkpoints' / 'best_model.pt')
VOCAB_FILE  = str(PROJECT_DIR / 'data' / 'process_data' / 'vocab.json')
LABEL_FILE  = str(PROJECT_DIR / 'data' / 'process_data' / 'label_map.json')
STOPWORDS_FILE = str(PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords.txt')

# ── Load files ───────────────────────────────────────────────────────────
with open(VOCAB_FILE,  encoding='utf-8') as f: vocab     = json.load(f)
with open(LABEL_FILE,  encoding='utf-8') as f: label_map = json.load(f)
with open(STOPWORDS_FILE, encoding='utf-8') as f:
    STOPWORDS = {line.strip() for line in f if line.strip()}

# idx → tên nhãn (để hiển thị kết quả)
IDX_TO_LABEL = {
    level: {v: k for k, v in label_map[level].items()}
    for level in ['l1', 'l2', 'l3']
}
NUM_CLASSES = [len(label_map['l1']), len(label_map['l2']), len(label_map['l3'])]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Vocab  : {len(vocab):,} từ')
print(f'Nhãn   : L1={NUM_CLASSES[0]}  L2={NUM_CLASSES[1]}  L3={NUM_CLASSES[2]}')

# ── Định nghĩa lại HARNN (copy từ Cell 4 notebook train) ─────────────────
class HARNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size,
                 num_classes_per_level, dropout=0.5):
        super().__init__()
        self.num_levels  = len(num_classes_per_level)
        self.hidden_size = hidden_size
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bigru       = nn.GRU(embed_dim, hidden_size,
                                  bidirectional=True, batch_first=True)
        self.dropout     = nn.Dropout(dropout)
        self.attention   = nn.ModuleList([
            nn.Linear(hidden_size * 2, 1) for _ in range(self.num_levels)
        ])
        self.ham = nn.LSTMCell(hidden_size * 2, hidden_size)
        self.classifiers = nn.ModuleList([
            nn.Linear(hidden_size * 3, n) for n in num_classes_per_level
        ])

    def forward(self, x):
        B    = x.size(0)
        emb  = self.dropout(self.embedding(x))
        doc, _ = self.bigru(emb)
        doc  = self.dropout(doc)
        h    = torch.zeros(B, self.hidden_size, device=x.device)
        c    = torch.zeros(B, self.hidden_size, device=x.device)
        preds = []
        for lv in range(self.num_levels):
            score   = self.attention[lv](doc)
            weight  = torch.softmax(score, dim=1)
            context = (weight * doc).sum(dim=1)
            h, c    = self.ham(context, (h, c))
            feat    = self.dropout(torch.cat([context, h], dim=-1))
            preds.append(torch.sigmoid(self.classifiers[lv](feat)))
        return preds

# ── Load checkpoint ───────────────────────────────────────────────────────
EMBED_DIM   = 100
HIDDEN_SIZE = 256

model = HARNN(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes_per_level=NUM_CLASSES,
).to(device)

ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'\nLoaded checkpoint epoch {ckpt["epoch"]}'
      f' (val F1-L1={ckpt["val_f1_l1"]:.3f})')


Device : cuda
Vocab  : 34,647 từ
Nhãn   : L1=13  L2=49  L3=31

Loaded checkpoint epoch 14 (val F1-L1=0.912)


## Cell 2 — Hàm predict
Nhận vào text, trả về nhãn kèm xác suất cho từng level.


In [11]:
# ============================================================
# CELL 2 — Hàm predict
# ============================================================
MAX_LEN   = 512
THRESHOLD = 0.5   # Hạ xuống 0.3 nếu muốn recall cao hơn

def clean_text(text: str) -> str:
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def tokenize(text: str) -> list[str]:
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean_text(text), format='text').split()
    except ImportError:
        tokens = clean_text(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def text_to_tensor(text: str) -> torch.Tensor:
    tokens = tokenize(text)
    ids    = [vocab.get(t, 1) for t in tokens][:MAX_LEN]
    ids   += [0] * (MAX_LEN - len(ids))
    return torch.tensor([ids], dtype=torch.long).to(device)   # (1, MAX_LEN)

@torch.no_grad()
def predict(
    text: str,
    threshold: float = THRESHOLD,
    top_k: int = 3,       # nếu không có nhãn nào qua threshold → lấy top_k
) -> dict:
    """
    Dự đoán nhãn cho một đoạn văn bản.

    Trả về dict:
    {
        'l1': [{'label': 'Thể thao', 'prob': 0.97}],
        'l2': [{'label': 'Bóng đá',  'prob': 0.89}],
        'l3': [],
        'tokens': ['mbappe', 'ghi_bàn', ...]
    }
    """
    x      = text_to_tensor(text)
    preds  = model(x)          # list of (1, num_classes_i)
    tokens = tokenize(text)

    result = {'tokens': tokens[:10]}

    for i, level in enumerate(['l1', 'l2', 'l3']):
        probs     = preds[i][0].cpu().numpy()          # (num_classes,)
        idx2label = IDX_TO_LABEL[level]

        # Lấy nhãn vượt threshold
        selected = [
            {'label': idx2label[j], 'prob': float(probs[j])}
            for j in range(len(probs))
            if probs[j] >= threshold
        ]

        # Nếu không có nhãn nào → lấy top_k nhãn cao nhất
        if not selected:
            top_indices = np.argsort(probs)[::-1][:top_k]
            selected    = [
                {'label': idx2label[j], 'prob': float(probs[j])}
                for j in top_indices
            ]

        # Sắp xếp theo prob giảm dần
        selected.sort(key=lambda x: x['prob'], reverse=True)
        result[level] = selected

    return result


def print_result(text: str, result: dict) -> None:
    """In kết quả đẹp ra màn hình."""
    print(f'{'─'*60}')
    print(f'Văn bản : {text[:80]}...' if len(text) > 80 else f'Văn bản : {text}')
    print(f'Tokens  : {result["tokens"]} ...')
    print()
    for level in ['l1', 'l2', 'l3']:
        labels = result[level]
        print(f'  {level.upper()}:')
        for item in labels:
            bar   = '█' * int(item['prob'] * 20)
            mark  = '✓' if item['prob'] >= THRESHOLD else '·'
            print(f'    {mark} {item["label"]:<25} {item["prob"]:>5.1%}  {bar}')
    print()

print('Hàm predict sẵn sàng!')
print(f'Threshold: {THRESHOLD}  (hạ xuống 0.3 nếu muốn recall cao hơn)')


Hàm predict sẵn sàng!
Threshold: 0.5  (hạ xuống 0.3 nếu muốn recall cao hơn)


In [12]:
# Chạy trong predict.ipynb để debug bài Mbappe
from pathlib import Path

text = "Mbappe lập hat-trick giúp Real Madrid đánh bại Barcelona 4-0 trong trận El Clasico"

# Xem token sau khi xử lý
tokens = tokenize(text)
print("Tokens:", tokens)

# Load Word2Vec (nếu có) để kiểm tra từ gần nghĩa
W2V_PATH = PROJECT_DIR / 'output' / 'models' / 'word2vec.model'
w2v = None
try:
    from gensim.models import Word2Vec
    if W2V_PATH.exists():
        w2v = Word2Vec.load(str(W2V_PATH))
        print(f"Loaded Word2Vec: {W2V_PATH}")
    else:
        print(f"Không tìm thấy file Word2Vec: {W2V_PATH}")
except Exception as e:
    print(f"Không load được Word2Vec ({e}). Bỏ qua phần similar words.")

# Xem token nào có trong vocab
for t in tokens:
    in_vocab = t in vocab
    if w2v is not None and t in w2v.wv:
        similar = w2v.wv.most_similar(t, topn=3)
        print(f"  {t:<20} vocab={in_vocab}  gần: {[w for w, _ in similar]}")
    else:
        status = "KHÔNG có trong W2V" if w2v is not None else "chưa load W2V"
        print(f"  {t:<20} vocab={in_vocab}  {status}")

Tokens: ['mbappe', 'lập', 'hat', 'trick', 'giúp', 'real', 'madrid', 'đánh_bại', 'barcelona', 'trận', 'el', 'clasico']
Loaded Word2Vec: C:\Users\Admin\Documents\nlp\NLP_Project\output\models\word2vec.model
  mbappe               vocab=True  gần: ['kylian', 'rodrygo', 'real']
  lập                  vocab=True  gần: ['trích_đo', 'kỷ_lục', 'doelon']
  hat                  vocab=True  gần: ['trick', 'hat_trick', 'ueki']
  trick                vocab=True  gần: ['hat', 'hat_trick', 'ueki']
  giúp                 vocab=True  gần: ['tối_ưu', 'góp_phần', 'hiệu_quả']
  real                 vocab=True  gần: ['la_liga', 'benfica', 'psg']
  madrid               vocab=True  gần: ['real', 'bernabeu', 'metropolitano']
  đánh_bại             vocab=True  gần: ['maestrelli', 'borges', 'lehecka']
  barcelona            vocab=True  gần: ['catalunya', 'silverstone', 'el_clasico']
  trận                 vocab=True  gần: ['đấu', 'thua', 'toàn_thắng']
  el                   vocab=True  gần: ['mencho', 'ẩn_dật',

## Cell 3 — Demo dự đoán
Thử với một số bài viết mẫu từ nhiều domain khác nhau.


In [14]:
# ============================================================
# CELL 3 — Demo dự đoán với bài viết mẫu
# Thay text bằng bài viết của bạn để test
# ============================================================

samples = [
    # Thể thao - Bóng đá
    ('Thể thao',
     'NMbappe lập hat-trick giúp Real Madrid đánh bại Barcelona 4-0 trong trận El Clasico'
     'trong trận El Clasico tại sân Bernabeu tối qua. '
     'Đây là chiến thắng thuyết phục nhất của Los Blancos trong thập kỷ này.'),

    # Sức khỏe
    ('Sức khỏe',
     'Bác sĩ khuyến cáo người bệnh tiểu đường cần kiểm soát đường huyết '
     'hàng ngày và duy trì chế độ ăn ít tinh bột. '
     'Tập thể dục đều đặn 30 phút mỗi ngày giúp cải thiện tình trạng bệnh.'),

    # Kinh doanh
    ('Kinh doanh',
     'Giá vàng SJC hôm nay tăng mạnh lên 85 triệu đồng mỗi lượng, '
     'cao nhất trong lịch sử. Các chuyên gia nhận định xu hướng tăng '
     'sẽ còn tiếp tục trong quý tới do nhu cầu trú ẩn tài sản an toàn.'),

    # Thời sự
    ('Thời sự',
     'Thủ tướng chủ trì hội nghị về phát triển kinh tế vùng đồng bằng sông Cửu Long. '
     'Nhiều giải pháp ứng phó biến đổi khí hậu và phát triển nông nghiệp bền vững '
     'đã được các địa phương đề xuất trong buổi làm việc.'),

    # Du lịch
    ('Du lịch',
     'Hội An lọt top 10 điểm đến hấp dẫn nhất châu Á năm 2024 theo bình chọn '
     'của tạp chí Travel + Leisure. Phố cổ với những ngôi nhà cổ kính và đèn lồng '
     'rực rỡ thu hút hàng triệu du khách quốc tế mỗi năm.'),
]

for expected, text in samples:
    result = predict(text)
    print_result(text, result)
    top_l1 = result['l1'][0]['label'] if result['l1'] else '?'
    correct = '✓ ĐÚNG' if top_l1 == expected else f'✗ SAI (đúng: {expected})'
    print(f'  → Dự đoán: {top_l1}  {correct}')
    print()


────────────────────────────────────────────────────────────
Văn bản : NMbappe lập hat-trick giúp Real Madrid đánh bại Barcelona 4-0 trong trận El Clas...
Tokens  : ['nmbappe', 'lập', 'hat', 'trick', 'giúp', 'real', 'madrid', 'đánh_bại', 'barcelona', 'trận'] ...

  L1:
    ✓ Thể thao                  100.0%  ███████████████████
  L2:
    ✓ Bóng đá                   99.1%  ███████████████████
  L3:
    ✓ Các giải khác             79.2%  ███████████████

  → Dự đoán: Thể thao  ✓ ĐÚNG

────────────────────────────────────────────────────────────
Văn bản : Bác sĩ khuyến cáo người bệnh tiểu đường cần kiểm soát đường huyết hàng ngày và d...
Tokens  : ['bác_sĩ', 'khuyến_cáo', 'người_bệnh', 'tiểu_đường', 'kiểm_soát', 'đường_huyết', 'hàng', 'duy_trì', 'chế_độ', 'tinh_bột'] ...

  L1:
    ✓ Sức khỏe                  99.9%  ███████████████████
  L2:
    ✓ Các bệnh                  99.5%  ███████████████████
  L3:
    ✓ Nội tiết                  91.1%  ██████████████████
    ✓ Tư vấn              

## Cell 4 — Predict bài viết của bạn
Nhập text trực tiếp hoặc đọc từ file để dự đoán hàng loạt.


In [15]:
# ============================================================
# CELL 4A — Nhập text trực tiếp
# Thay YOUR_TEXT bằng bài viết muốn dự đoán
# ============================================================

YOUR_TEXT = """
Sáng 20/3, Ủy ban bầu cử TP Hà Nội công bố kết quả bầu cử và danh sách 125 người trúng cử đại biểu HĐND thành phố nhiệm kỳ 2026-2031.

Danh sách trúng cử gồm nhiều lãnh đạo chủ chốt như Chủ tịch UBND thành phố Vũ Đại Thắng, Chủ tịch HĐND thành phố Phùng Thị Hồng Hà, Phó bí thư Thường trực Thành ủy Hà Nội Nguyễn Trọng Đông, Phó chủ tịch Thường trực HĐND TP Hà Nội Trần Thế Cương, Giám đốc Công an TP Hà Nội Nguyễn Thanh Tùng, Tư lệnh Bộ Tư lệnh Thủ đô Hà Nội Đào Văn Nhận.

Một số doanh nhân cũng trúng cử như ông Đỗ Vinh Quang, Phó chủ tịch Hội đồng quản trị kiêm Phó tổng giám đốc Công ty Cổ phần Tập đoàn T&T; ông Nguyễn Duy Chính, Tổng giám đốc Công ty Cổ phần Tập đoàn Tân Á Đại Thành; ông Phạm Tiến Đức, Tổng giám đốc Tổng công ty Đầu tư và Phát triển nhà Hà Nội.

Danh sách trúng cử còn có đại diện các lĩnh vực văn hóa, giáo dục, báo chí như ông Nguyễn Trung Hiếu, Giám đốc Nhà hát Kịch Hà Nội; bà Nguyễn Thị Nhiếp, Hiệu trưởng Trường trung học phổ thông chuyên Chu Văn An; ông Vũ Minh Tuấn, Giám đốc cơ quan Báo và Phát thanh, Truyền hình Hà Nội.
"""

if YOUR_TEXT.strip() and 'Paste' not in YOUR_TEXT:
    result = predict(YOUR_TEXT)
    print_result(YOUR_TEXT, result)
else:
    print('Thay YOUR_TEXT bằng nội dung bài viết của bạn')


────────────────────────────────────────────────────────────
Văn bản : 
Sáng 20/3, Ủy ban bầu cử TP Hà Nội công bố kết quả bầu cử và danh sách 125 ngườ...
Tokens  : ['ủy', 'ban', 'bầu_cử', 'tp', 'hà_nội', 'công_bố', 'kết_quả', 'bầu_cử', 'danh_sách', 'trúng_cử'] ...

  L1:
    ✓ Thời sự                   99.8%  ███████████████████
  L2:
    ✓ Chính trị                 76.1%  ███████████████
  L3:
    · Kỹ năng lái                0.2%  
    · Đại học                    0.0%  
    · Trong nước                 0.0%  



In [16]:
# ============================================================
# CELL 4B — Batch predict, hiển thị đầy đủ bằng DataFrame
# ============================================================
import pandas as pd
pd.set_option('display.max_rows', True)
pd.set_option('display.max_colwidth', 60)

RAW_FILE_FOR_PREDICT = str(PROJECT_DIR / 'data' / 'raw' / 'raw_data.json')

with open(RAW_FILE_FOR_PREDICT, encoding='utf-8') as f:
    raw_all = json.load(f)
raw_dict = {a['url']: a for a in raw_all}

with open(str(PROJECT_DIR / 'data' / 'process_data' / 'dataset.json'), 'r', encoding='utf-8') as f:
          encoding='utf-8') as f:
    dataset = json.load(f)

# Đổi số lượng bài ở đây
samples = dataset[-200:]   # ← thay 100 bằng bất kỳ số nào

rows = []
for a in samples:
    raw  = raw_dict.get(a['url'], {})
    text = raw.get('title', '') + ' ' + raw.get('content', '')
    if not text.strip():
        text = a['title'] + ' ' + ' '.join(a['tokens'])

    result = predict(text, threshold=0.5)

    true_l1 = a['labels_l1'][0] if a['labels_l1'] else ''
    true_l2 = a['labels_l2'][0] if a['labels_l2'] else ''
    pred_l1 = result['l1'][0]['label'] if result['l1'] else ''
    pred_l2 = result['l2'][0]['label'] if result['l2'] else ''
    prob_l1 = result['l1'][0]['prob']  if result['l1'] else 0
    prob_l2 = result['l2'][0]['prob']  if result['l2'] else 0

    rows.append({
        'Title':      a['title'][:60],
        'L1 đúng':    true_l1,
        'L1 dự đoán': pred_l1,
        'Prob L1':    f'{prob_l1:.2f}',
        'OK L1':      '✓' if true_l1 == pred_l1 else '✗',
        'L2 đúng':    true_l2,
        'L2 dự đoán': pred_l2,
        'Prob L2':    f'{prob_l2:.2f}',
        'OK L2':      '✓' if (true_l2 and true_l2 == pred_l2) else ('—' if not true_l2 else '✗'),
    })

df = pd.DataFrame(rows)

# Tô màu cột OK
def color_ok(val):
    if val == '✓': return 'background-color: #d4edda; color: #155724'
    if val == '✗': return 'background-color: #f8d7da; color: #721c24'
    return ''

display(df.style.applymap(color_ok, subset=['OK L1', 'OK L2']))

# Thống kê
total   = len(df)
acc_l1  = (df['OK L1'] == '✓').sum()
has_l2  = (df['L2 đúng'] != '').sum()
acc_l2  = ((df['OK L2'] == '✓')).sum()

print(f'\nAccuracy L1: {acc_l1}/{total} ({acc_l1/total*100:.1f}%)')
print(f'Accuracy L2: {acc_l2}/{has_l2} ({acc_l2/has_l2*100:.1f}%)  [chỉ tính bài có L2 gốc]')

C:\Users\Admin\AppData\Local\Temp\ipykernel_19940\1388611280.py:57: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(df.style.applymap(color_ok, subset=['OK L1', 'OK L2']))


,Title,L1 đúng,L1 dự đoán,Prob L1,OK L1,L2 đúng,L2 dự đoán,Prob L2,OK L2
0,Radar chống UAV ở đại sứ quán Mỹ cháy đen sau 'đòn tập kích,Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.98,✓
1,Vua và Hoàng hậu Thái Lan tự lái chuyên cơ thăm Lào,Thế giới,Thế giới,1.00,✓,Cuộc sống đó đây,Cuộc sống đó đây,0.88,✓
2,UAV tập kích đại sứ quán Mỹ tại Arab Saudi,Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.45,✓
3,Ông Trump: Chiến dịch tấn công Iran có thể kéo dài 4-5 tuần,Thế giới,Thế giới,1.00,✓,,Tư liệu,0.11,—
4,Ukraine đăng video tên lửa Storm Shadow lao xuống 'nhà máy N,Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.96,✓
5,Australia điều 'máy bay mắt thần' chuyên chống UAV đến Trung,Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.94,✓
6,"Mỹ tốn 5,6 tỷ USD đạn dược trong hai ngày đầu tấn công Iran",Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.89,✓
7,"Bangladesh cho sinh viên nghỉ học để tiết kiệm điện, xăng dầ",Thế giới,Thế giới,1.00,✓,Cuộc sống đó đây,Cuộc sống đó đây,0.83,✓
8,Mỹ nêu lý do cùng Israel không kích Iran,Thế giới,Thế giới,1.00,✓,Quân sự,Quân sự,0.55,✓
9,4 nhà ngoại giao Iran thiệt mạng trong đòn không kích ở Leba,Thế giới,Thế giới,1.00,✓,,Quân sự,0.33,—



Accuracy L1: 199/200 (99.5%)
Accuracy L2: 113/125 (90.4%)  [chỉ tính bài có L2 gốc]
